In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

def evaluate_baseline(train_df, test_df, model, model_name):
    print(f"\n====== BASELINE ({model_name}) ======\n")

    features = train_df.drop(columns=["label","apkname","year"], errors="ignore").select_dtypes(exclude=["object"]).columns

    X_train = train_df[features]
    y_train = train_df["label"]

    X_test = test_df[features]
    y_test = test_df["label"]

    # scaling
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # fit
    model.fit(X_train_s, y_train)

    # predict
    prob = model.predict_proba(X_test_s)[:,1]
    pred = (prob >= 0.5).astype(int)

    # overall
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    try:
        auc = roc_auc_score(y_test, prob)
    except:
        auc = np.nan
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    fpr = fp / (fp + tn)

    print(f"Overall ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}, FPR={fpr:.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    # year-wise
    year_rows = []
    years = sorted(test_df["year"].dropna().unique())

    print("\n--- YEAR-WISE ---")

    for y in years:
        df_y = test_df[test_df["year"] == y]

        # 🔥 empty year skip
        if len(df_y) == 0:
            print(f"Year {y}: NO SAMPLES → skipped")
            continue

        Xy = scaler.transform(df_y[features])
        yy = df_y["label"]

        prob_y = model.predict_proba(Xy)[:,1]
        pred_y = (prob_y >= 0.5).astype(int)

        acc_y = accuracy_score(yy, pred_y)
        f1_y = f1_score(yy, pred_y)
        try:
            auc_y = roc_auc_score(yy, prob_y)
        except:
            auc_y = np.nan
        tn, fp, fn, tp = confusion_matrix(yy, pred_y).ravel()
        fpr_y = fp / (fp + tn)

        year_rows.append([y, acc_y, f1_y, auc_y, fpr_y])

    print(pd.DataFrame(year_rows, columns=["Year","Accuracy","F1","AUC","FPR"]).to_string(index=False))


def run_baseline_all(train_df, test_df, test_name):
    print(f"\n\n================ BASELINE: {test_name} ================\n")

    models = {
        "RF": RandomForestClassifier(),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "SVM": SVC(probability=True),
        "XGB": XGBClassifier(eval_metric="logloss", use_label_encoder=False)
    }

    for mname, model in models.items():
        evaluate_baseline(train_df, test_df, model, mname)



In [2]:
def run_fSCAN(train_df, test_df, name):
    log(f"\n============ f-SCAN {name} START ============")

    gmm, scaler, gmm_train_df, X_scaled = fit_gmm_and_scaler(
        train_df,
        ["apkname","label","year"],
        19
    )

    # prune
    gmm_train_df, alive, deleted = prune_and_reassign_clusters(
        gmm_train_df, gmm, X_scaled
    )

    print_cluster_stats(gmm_train_df)

    # cluster-wise models (RF)
    rf_models = train_cluster_rf_models(gmm_train_df, 19)

    # ⬅ Threshold fixed to 0.5
    th = np.array([0.5] * 19)

    mapped = build_mapped_sets_yearwise(
        test_df,
        scaler,
        gmm,
        ["apkname","label","year"],
        years=[2018,2019,2020,2021,2022,2023]
    )

    overall, df_year = evaluate(mapped, rf_models, th)

    if overall is None:
        log("⚠ No predictions.\n")
        return

    acc,f1,auc,fpr,tn,fp,fn,tp = overall

    log(f"\n===== f-SCAN {name} RESULTS =====")
    log(f"ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}, FPR={fpr:.4f}")
    log(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print("\n--- Year-wise ---")
    print(df_year.to_string(index=False))

    log(f"\n============ f-SCAN {name} END ============\n")


In [3]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# Utility
# ============================================================
def log(s):
    print(s, flush=True)


# ============================================================
# SCAN Functions (원본 그대로)
# ============================================================

def fit_gmm_and_scaler(train_df, features_to_remove, cluster_num, seed=42):
    log("Extracting GMM training data (2014–2017)...")
    gmm_train_df = train_df.copy()

    X = gmm_train_df.drop(columns=features_to_remove, errors="ignore") \
                    .select_dtypes(exclude=["object"])

    log("Scaling features…")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    log(f"Training GMM K={cluster_num} (diag covariance)…")
    gmm = GaussianMixture(
        n_components=cluster_num,
        covariance_type="diag",
        random_state=seed
    )
    gmm.fit(X_scaled)

    gmm_train_df["cluster"] = gmm.predict(X_scaled)
    return gmm, scaler, gmm_train_df, X_scaled


def prune_and_reassign_clusters(gmm_train_df, gmm, X_scaled, min_samples=20):
    log("\n=== Cluster Pruning ===")

    df_cnt = gmm_train_df.groupby(["cluster", "label"]).size().unstack(fill_value=0)
    df_cnt["Total"] = df_cnt.sum(axis=1)

    deleted = []
    alive = []

    for k in range(gmm.n_components):
        if k not in df_cnt.index:
            deleted.append(k)
            continue

        benign = df_cnt.loc[k, 0] if 0 in df_cnt.columns else 0
        mal = df_cnt.loc[k, 1] if 1 in df_cnt.columns else 0
        tot = benign + mal

        if tot < min_samples or benign == 0 or mal == 0:
            deleted.append(k)
        else:
            alive.append(k)

    log(f"Deleted clusters: {deleted}")
    log(f"Alive clusters:   {alive}")

    # Reassignment
    R = gmm.predict_proba(X_scaled)
    new_clusters = gmm_train_df["cluster"].values.copy()
    alive_arr = np.array(alive)

    for i in range(len(new_clusters)):
        if new_clusters[i] in deleted:
            probs = R[i, alive_arr]
            new_clusters[i] = alive_arr[np.argmax(probs)]

    gmm_train_df["cluster"] = new_clusters
    return gmm_train_df, alive, deleted


def train_cluster_rf_models(gmm_train_df, cluster_num, seed=42):
    log("\nTraining RF models per cluster…")
    models = [None] * cluster_num

    for k in range(cluster_num):
        dfc = gmm_train_df[gmm_train_df["cluster"] == k]
        if len(dfc) < 2:
            continue

        X = dfc.drop(columns=["cluster","label","year","apkname"], errors="ignore") \
               .select_dtypes(exclude=["object"])
        y = dfc["label"]
        y_cnt = y.value_counts()

        try:
            if len(y_cnt) < 2 or y_cnt.min() < 2:
                X_tr, _, y_tr, _ = train_test_split(X, y, test_size=0.2, random_state=seed)
            else:
                X_tr, _, y_tr, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
        except:
            continue

        model = RandomForestClassifier(random_state=seed)
        model.fit(X_tr, y_tr)
        models[k] = model

    return models


def compute_adaptive_thresholds(gmm_train_df, cluster_num, alpha):
    dist = gmm_train_df.groupby(["cluster","label"]).size().unstack(fill_value=0)
    dist.columns = ["Benign","Malicious"]
    dist["Total"] = dist.sum(axis=1)
    dist["Ratio"] = dist["Malicious"] / dist["Total"]

    r = dist["Ratio"].reindex(range(cluster_num)).fillna(0.5).values
    d = np.sqrt((2*alpha-1)**2 + (2*r-1)**2)
    u = d / np.sqrt(2)

    th = 0.5 + 0.5*np.sqrt(u)
    th = np.clip(th, 0.5, 0.98)

    return th, dist


def build_mapped_sets_yearwise(test_df, scaler, gmm, features_to_remove, years):
    out = {}
    for y in years:
        df = test_df[test_df["year"] == y].copy()
        if df.empty:
            continue

        X = df.drop(columns=features_to_remove, errors="ignore") \
              .select_dtypes(exclude=["object"])
        Xs = scaler.transform(X)

        df["cluster"] = gmm.predict(Xs)
        out[y] = df
    return out


def evaluate(mapped, rf_models, th):
    total_preds, total_labels, total_prob = [], [], []
    rows = []

    for year, df in sorted(mapped.items()):
        yp, yt, yp_prob = [], [], []

        for k in df["cluster"].unique():
            sub = df[df["cluster"] == k]
            if sub.empty:
                continue
            if rf_models[k] is None:
                continue

            feats = rf_models[k].feature_names_in_
            Xv = sub[feats]
            yv = sub["label"]

            prob = rf_models[k].predict_proba(Xv)[:,1]
            pred = (prob >= th[k]).astype(int)

            yp.extend(pred)
            yt.extend(yv)
            yp_prob.extend(prob)

            total_preds.extend(pred)
            total_labels.extend(yv)
            total_prob.extend(prob)

        if len(yt) > 0:
            acc = accuracy_score(yt, yp)
            f1  = f1_score(yt, yp)
            try:
                auc = roc_auc_score(yt, yp_prob)
            except:
                auc = np.nan
            tn,fp,fn,tp = confusion_matrix(yt, yp).ravel()
            fpr = fp/(fp+tn)
            rows.append([year, acc, f1, auc, fpr])

    if len(total_labels)==0:
        return None, None

    acc = accuracy_score(total_labels, total_preds)
    f1  = f1_score(total_labels, total_preds)
    try:
        auc = roc_auc_score(total_labels, total_prob)
    except:
        auc = np.nan
    tn,fp,fn,tp = confusion_matrix(total_labels, total_preds).ravel()
    fpr = fp/(fp+tn)

    df_year = pd.DataFrame(rows, columns=["Year","Accuracy","F1","AUC","FPR"])
    return (acc,f1,auc,fpr,tn,fp,fn,tp), df_year


def print_cluster_stats(df):
    tab = df.groupby(["cluster","label"]).size().unstack(fill_value=0)
    tab.columns=["Benign","Malicious"]
    tab["Total"]=tab.sum(axis=1)

    log("\n=== Cluster Stats ===")
    print(tab.to_string())
    log("=====================")


# ============================================================
# SCAN per Test Set
# ============================================================

def run_SCAN(train_df, test_df, name):
    log(f"\n============ {name} START ============")

    gmm, scaler, gmm_train_df, X_scaled = fit_gmm_and_scaler(
        train_df,
        ["apkname","label","year"],
        19
    )

    gmm_train_df, alive, deleted = prune_and_reassign_clusters(
        gmm_train_df, gmm, X_scaled
    )

    print_cluster_stats(gmm_train_df)

    rf_models = train_cluster_rf_models(gmm_train_df, 19)

    th, _ = compute_adaptive_thresholds(gmm_train_df, 19, alpha=0.5)

    mapped = build_mapped_sets_yearwise(
        test_df,
        scaler,
        gmm,
        ["apkname","label","year"],
        years=[2018,2019,2020,2021,2022,2023]
    )

    overall, df_year = evaluate(mapped, rf_models, th)

    if overall is None:
        log("⚠ No valid predictions.\n")
        return

    acc,f1,auc,fpr,tn,fp,fn,tp = overall

    log(f"\n===== {name} RESULTS =====")
    log(f"ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}, FPR={fpr:.4f}")
    log(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    log("\n--- Year-wise ---")
    print(df_year.to_string(index=False))

    log(f"\n============ {name} END ============\n")

In [4]:
train_df = pd.read_csv("CV_train_data.csv")
test1_df = pd.read_csv("CV_test1_data.csv")
test2_df = pd.read_csv("CV_test2_data.csv")

# Baseline
run_baseline_all(train_df, test1_df, "TEST1 (1:1)")
run_baseline_all(train_df, test2_df, "TEST2 (9:1)")

# f-SCAN
run_fSCAN(train_df, test1_df, "TEST1 (1:1)")
run_fSCAN(train_df, test2_df, "TEST2 (9:1)")

# SCAN (original)
run_SCAN(train_df, test1_df, "TEST1 (1:1)")
run_SCAN(train_df, test2_df, "TEST2 (9:1)")




================ BASELINE: TEST1 (1:1) ================


====== BASELINE (RF) ======

Overall ACC=0.6287, F1=0.7279, AUC=0.9715, FPR=0.7362
Confusion Matrix: TN=1467, FP=4093, FN=36, TP=5524

--- YEAR-WISE ---
  Year  Accuracy       F1      AUC      FPR
2018.0  0.951764 0.930426 0.977374 0.056156
2019.0  0.466187 0.554087 0.968922 0.798274
2020.0  0.413967 0.531646 0.966980 0.877970
2021.0  0.417986 0.525513 0.909770 0.856526
2022.0  0.404745 0.527936 0.985120 0.892125
2023.0  0.375270 0.515337 0.943230 0.935275

====== BASELINE (KNN) ======

Overall ACC=0.8862, F1=0.8908, AUC=0.9158, FPR=0.1552
Confusion Matrix: TN=4697, FP=863, FN=402, TP=5158

--- YEAR-WISE ---
  Year  Accuracy       F1      AUC      FPR
2018.0  0.919366 0.884536 0.943963 0.084233
2019.0  0.850360 0.809524 0.868944 0.201726
2020.0  0.881929 0.836327 0.891740 0.129590
2021.0  0.751799 0.586826 0.667683 0.137001
2022.0  0.834651 0.797178 0.867430 0.235167
2023.0  0.877786 0.833984 0.926484 0.143474

====== BASELINE

Scaling features…
Training GMM K=19 (diag covariance)…

=== Cluster Pruning ===
Deleted clusters: [1, 8, 12, 13]
Alive clusters:   [0, 2, 3, 4, 5, 6, 7, 9, 10, 11, 14, 15, 16, 17, 18]

=== Cluster Stats ===
         Benign  Malicious  Total
cluster                          
0          1673       2862   4535
2           437        290    727
3            77       1089   1166
4           349        109    458
5           104         21    125
6           359         57    416
7            33         57     90
9           117        264    381
10          399        101    500
11           53        165    218
14          287        213    500
15          128         43    171
16          219        142    361
17          124         76    200
18         1201         71   1272

Training RF models per cluster…

===== TEST2 (9:1) RESULTS =====
ACC=0.9395, F1=0.5916, AUC=0.9436, FPR=0.0544
Confusion Matrix: TN=16972, FP=976, FN=170, TP=830

--- Year-wise ---
 Year  Accuracy       F1      AUC

In [8]:
import pandas as pd
import numpy as np
import warnings
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# Utility
# ============================================================
def log(s):
    print(s, flush=True)

# ============================================================
# Sanitize Feature Names  (XGB 필수)
# ============================================================
def sanitize_feature_names(df):
    df = df.copy()
    new_cols = []
    for c in df.columns:
        nc = (
            c.replace("[", "_")
             .replace("]", "_")
             .replace("<", "_")
             .replace(">", "_")
             .replace(" ", "_")
             .replace(",", "_")
        )
        new_cols.append(nc)
    df.columns = new_cols
    return df

# ============================================================
# GMM + Scaling
# ============================================================
def fit_gmm_and_scaler(train_df, features_to_remove, cluster_num, seed=42):
    log("Extracting GMM training data (2014–2017)...")
    gmm_train_df = train_df.copy()

    X = gmm_train_df.drop(columns=features_to_remove, errors="ignore") \
                    .select_dtypes(exclude=["object"])

    log("Scaling features…")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    log(f"Training GMM K={cluster_num} (diag covariance)…")
    gmm = GaussianMixture(
        n_components=cluster_num,
        covariance_type="diag",
        random_state=seed
    )
    gmm.fit(X_scaled)

    gmm_train_df["cluster"] = gmm.predict(X_scaled)
    return gmm, scaler, gmm_train_df, X_scaled


# ============================================================
# Cluster pruning + reassignment
# ============================================================
def prune_and_reassign_clusters(gmm_train_df, gmm, X_scaled, min_samples=20):
    log("\n=== Cluster Pruning ===")
    df_cnt = gmm_train_df.groupby(["cluster", "label"]).size().unstack(fill_value=0)
    df_cnt["Total"] = df_cnt.sum(axis=1)

    deleted = []
    alive = []
    for k in range(gmm.n_components):
        if k not in df_cnt.index:
            deleted.append(k)
            continue
        benign = df_cnt.loc[k, 0] if 0 in df_cnt.columns else 0
        mal    = df_cnt.loc[k, 1] if 1 in df_cnt.columns else 0
        tot = benign + mal
        if tot < min_samples or benign == 0 or mal == 0:
            deleted.append(k)
        else:
            alive.append(k)

    log(f"Deleted clusters: {deleted}")
    log(f"Alive clusters:   {alive}")

    # reassignment
    R = gmm.predict_proba(X_scaled)
    alive_arr = np.array(alive)
    new_clusters = gmm_train_df["cluster"].values.copy()

    for i in range(len(new_clusters)):
        if new_clusters[i] in deleted:
            probs = R[i, alive_arr]
            new_clusters[i] = alive_arr[np.argmax(probs)]

    gmm_train_df["cluster"] = new_clusters
    return gmm_train_df, alive, deleted


# ============================================================
# Model maker (RF / SVM / KNN / XGB 지원)
# ============================================================
def make_model(model_type):
    if model_type == "RF":
        return RandomForestClassifier(random_state=42)
    if model_type == "KNN":
        return KNeighborsClassifier(n_neighbors=5)
    if model_type == "SVM":
        return SVC(probability=True)
    if model_type == "XGB":
        return XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            n_jobs=-1,
            tree_method="auto"
        )
    raise ValueError("Unknown model type")


# ============================================================
# Train model per cluster
# ============================================================
def train_cluster_models(gmm_train_df, cluster_num, model_type):
    log(f"Training cluster models ({model_type})…")
    models = [None] * cluster_num

    for k in range(cluster_num):
        dfc = gmm_train_df[gmm_train_df["cluster"] == k]
        if len(dfc) < 3:
            continue

        X = dfc.drop(columns=["cluster","label","year","apkname"], errors="ignore") \
               .select_dtypes(exclude=["object"])
        y = dfc["label"]

        y_cnt = y.value_counts()

        try:
            if len(y_cnt) < 2 or y_cnt.min() < 2:
                Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.2)
            else:
                Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.2, stratify=y)
        except:
            continue

        model = make_model(model_type)
        model.fit(Xtr.values, ytr)
        models[k] = model

    return models


# ============================================================
# Adaptive threshold (SCAN)
# ============================================================
def compute_adaptive_thresholds(gmm_train_df, cluster_num, alpha):
    dist = gmm_train_df.groupby(["cluster","label"]).size().unstack(fill_value=0)
    dist.columns = ["Benign","Malicious"]
    dist["Total"] = dist.sum(axis=1)
    dist["Ratio"] = dist["Malicious"] / dist["Total"]

    r = dist["Ratio"].reindex(range(cluster_num)).fillna(0.5).values
    d = np.sqrt((2*alpha-1)**2 + (2*r-1)**2)
    u = d / np.sqrt(2)

    th = 0.5 + 0.5*np.sqrt(u)
    th = np.clip(th, 0.5, 0.98)
    return th, dist


# ============================================================
# Map test samples to clusters
# ============================================================
def build_mapped_sets_yearwise(test_df, scaler, gmm, features_to_remove, years):
    out = {}
    for y in years:
        df = test_df[test_df["year"] == y].copy()
        if df.empty:
            continue
        X = df.drop(columns=features_to_remove, errors="ignore") \
              .select_dtypes(exclude=["object"])
        Xs = scaler.transform(X)
        df["cluster"] = gmm.predict(Xs)
        out[y] = df
    return out


# ============================================================
# Evaluation
# ============================================================
def evaluate(mapped, models, th):
    total_preds, total_labels, total_probs = [], [], []
    rows = []

    for year, df in sorted(mapped.items()):
        yp, yt, yp_prob = [], [], []

        for k in df["cluster"].unique():
            sub = df[df["cluster"] == k]
            if sub.empty:
                continue
            model = models[k]
            if model is None:
                continue

            feats = sub[model.feature_names_in_] if hasattr(model, "feature_names_in_") else \
                    sub.drop(columns=["cluster","label","year","apkname"], errors="ignore") \
                       .select_dtypes(exclude=["object"])
            prob = model.predict_proba(feats.values)[:,1]
            pred = (prob >= th[k]).astype(int)

            yp.extend(pred)
            yt.extend(sub["label"])
            yp_prob.extend(prob)

            total_preds.extend(pred)
            total_labels.extend(sub["label"])
            total_probs.extend(prob)

        if len(yt) > 0:
            acc = accuracy_score(yt, yp)
            f1  = f1_score(yt, yp)
            try:
                auc = roc_auc_score(yt, yp_prob)
            except:
                auc = np.nan
            tn,fp,fn,tp = confusion_matrix(yt, yp).ravel()
            fpr = fp/(fp+tn)
            rows.append([year, acc, f1, auc, fpr])

    # overall
    acc = accuracy_score(total_labels, total_preds)
    f1  = f1_score(total_labels, total_preds)
    try:
        auc = roc_auc_score(total_labels, total_probs)
    except:
        auc = np.nan
    tn,fp,fn,tp = confusion_matrix(total_labels, total_preds).ravel()
    fpr = fp/(fp+tn)

    df_year = pd.DataFrame(rows, columns=["Year","Accuracy","F1","AUC","FPR"])
    return (acc,f1,auc,fpr,tn,fp,fn,tp), df_year


# ============================================================
# PRINT Cluster Stats
# ============================================================
def print_cluster_stats(df):
    tab = df.groupby(["cluster","label"]).size().unstack(fill_value=0)
    tab.columns=["Benign","Malicious"]
    tab["Total"] = tab.sum(axis=1)
    log(tab.to_string())


# ============================================================
# RUN: SCAN
# ============================================================
def run_SCAN(train_df, test_df, name, model_type):
    log(f"\n============ SCAN-{model_type} {name} START ============")

    gmm, scaler, gmm_df, Xs = fit_gmm_and_scaler(
        train_df, ["apkname","label","year"], 19
    )
    gmm_df, alive, deleted = prune_and_reassign_clusters(gmm_df, gmm, Xs)
    print_cluster_stats(gmm_df)

    models = train_cluster_models(gmm_df, 19, model_type)

    th, _ = compute_adaptive_thresholds(gmm_df, 19, alpha=0.5)

    mapped = build_mapped_sets_yearwise(
        test_df, scaler, gmm,
        ["apkname","label","year"],
        years=[2018,2019,2020,2021,2022,2023]
    )

    overall, df_year = evaluate(mapped, models, th)

    acc,f1,auc,fpr,tn,fp,fn,tp = overall
    log(f"\n===== SCAN-{model_type} {name} RESULTS =====")
    log(f"ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}, FPR={fpr:.4f}")
    log(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(df_year.to_string(index=False))
    log(f"============ SCAN-{model_type} {name} END ============\n")


# ============================================================
# RUN: f-SCAN
# ============================================================
def run_fSCAN(train_df, test_df, name, model_type):
    log(f"\n============ f-SCAN-{model_type} {name} START ============")

    gmm, scaler, gmm_df, Xs = fit_gmm_and_scaler(
        train_df, ["apkname","label","year"], 19
    )
    gmm_df, alive, deleted = prune_and_reassign_clusters(gmm_df, gmm, Xs)
    print_cluster_stats(gmm_df)

    models = train_cluster_models(gmm_df, 19, model_type)

    th = np.array([0.5] * 19)   # fixed threshold

    mapped = build_mapped_sets_yearwise(
        test_df, scaler, gmm,
        ["apkname","label","year"],
        years=[2018,2019,2020,2021,2022,2023]
    )

    overall, df_year = evaluate(mapped, models, th)

    acc,f1,auc,fpr,tn,fp,fn,tp = overall
    log(f"\n===== f-SCAN-{model_type} {name} RESULTS =====")
    log(f"ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}, FPR={fpr:.4f}")
    log(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(df_year.to_string(index=False))
    log(f"============ f-SCAN-{model_type} {name} END ============\n")


In [9]:
train_df = sanitize_feature_names(pd.read_csv("CV_train_data.csv"))
test1_df = sanitize_feature_names(pd.read_csv("CV_test1_data.csv"))

run_SCAN(train_df, test1_df, "TEST1", model_type="RF")
run_SCAN(train_df, test1_df, "TEST1", model_type="SVM")
run_SCAN(train_df, test1_df, "TEST1", model_type="KNN")
run_SCAN(train_df, test1_df, "TEST1", model_type="XGB")

run_fSCAN(train_df, test1_df, "TEST1", model_type="RF")
run_fSCAN(train_df, test1_df, "TEST1", model_type="SVM")
run_fSCAN(train_df, test1_df, "TEST1", model_type="KNN")
run_fSCAN(train_df, test1_df, "TEST1", model_type="XGB")



============ SCAN-RF TEST1 START ============
Extracting GMM training data (2014–2017)...
Scaling features…
Training GMM K=19 (diag covariance)…

=== Cluster Pruning ===
Deleted clusters: [1, 8, 12, 13]
Alive clusters:   [0, 2, 3, 4, 5, 6, 7, 9, 10, 11, 14, 15, 16, 17, 18]
         Benign  Malicious  Total
cluster                          
0          1673       2862   4535
2           437        290    727
3            77       1089   1166
4           349        109    458
5           104         21    125
6           359         57    416
7            33         57     90
9           117        264    381
10          399        101    500
11           53        165    218
14          287        213    500
15          128         43    171
16          219        142    361
17          124         76    200
18         1201         71   1272
Training cluster models (RF)…

===== SCAN-RF TEST1 RESULTS =====
ACC=0.9112, F1=0.8643, AUC=0.9480, FPR=0.0575
Confusion Matrix: TN=5226, FP=319, F


============ f-SCAN-KNN TEST1 START ============
Extracting GMM training data (2014–2017)...
Scaling features…
Training GMM K=19 (diag covariance)…

=== Cluster Pruning ===
Deleted clusters: [1, 8, 12, 13]
Alive clusters:   [0, 2, 3, 4, 5, 6, 7, 9, 10, 11, 14, 15, 16, 17, 18]
         Benign  Malicious  Total
cluster                          
0          1673       2862   4535
2           437        290    727
3            77       1089   1166
4           349        109    458
5           104         21    125
6           359         57    416
7            33         57     90
9           117        264    381
10          399        101    500
11           53        165    218
14          287        213    500
15          128         43    171
16          219        142    361
17          124         76    200
18         1201         71   1272
Training cluster models (KNN)…

===== f-SCAN-KNN TEST1 RESULTS =====
ACC=0.6964, F1=0.6600, AUC=0.8864, FPR=0.3975
Confusion Matrix: TN=3341, FP

In [12]:
train_df = sanitize_feature_names(train_df)
test1_df = sanitize_feature_names(test1_df)
test2_df = sanitize_feature_names(test2_df)

# ============================================
# Apply sanitize to ALL datasets
# ============================================
train_df = sanitize_feature_names(train_df)
test1_df = sanitize_feature_names(test1_df)
test2_df = sanitize_feature_names(test2_df)



# ============================================
# TEST2 실행 (SCAN + f-SCAN) 전체 자동 실행
# ============================================

log("\n================ RUNNING TEST2 (9:1) ================\n")

# ---- SCAN ----
run_SCAN(train_df, test2_df, "TEST2 (9:1)", model_type="RF")
run_SCAN(train_df, test2_df, "TEST2 (9:1)", model_type="SVM")
run_SCAN(train_df, test2_df, "TEST2 (9:1)", model_type="KNN")
run_SCAN(train_df, test2_df, "TEST2 (9:1)", model_type="XGB")

# ---- f-SCAN ----
run_fSCAN(train_df, test2_df, "TEST2 (9:1)", model_type="RF")
run_fSCAN(train_df, test2_df, "TEST2 (9:1)", model_type="SVM")
run_fSCAN(train_df, test2_df, "TEST2 (9:1)", model_type="KNN")
run_fSCAN(train_df, test2_df, "TEST2 (9:1)", model_type="XGB")

log("\n================ FINISHED TEST2 (9:1) ================\n")



================ RUNNING TEST2 (9:1) ================


============ SCAN-RF TEST2 (9:1) START ============
Extracting GMM training data (2014–2017)...
Scaling features…
Training GMM K=19 (diag covariance)…

=== Cluster Pruning ===
Deleted clusters: [1, 8, 12, 13]
Alive clusters:   [0, 2, 3, 4, 5, 6, 7, 9, 10, 11, 14, 15, 16, 17, 18]
         Benign  Malicious  Total
cluster                          
0          1673       2862   4535
2           437        290    727
3            77       1089   1166
4           349        109    458
5           104         21    125
6           359         57    416
7            33         57     90
9           117        264    381
10          399        101    500
11           53        165    218
14          287        213    500
15          128         43    171
16          219        142    361
17          124         76    200
18         1201         71   1272
Training cluster models (RF)…

===== SCAN-RF TEST2 (9:1) RESULTS =====
ACC=0.9394, F1


============ f-SCAN-KNN TEST2 (9:1) START ============
Extracting GMM training data (2014–2017)...
Scaling features…
Training GMM K=19 (diag covariance)…

=== Cluster Pruning ===
Deleted clusters: [1, 8, 12, 13]
Alive clusters:   [0, 2, 3, 4, 5, 6, 7, 9, 10, 11, 14, 15, 16, 17, 18]
         Benign  Malicious  Total
cluster                          
0          1673       2862   4535
2           437        290    727
3            77       1089   1166
4           349        109    458
5           104         21    125
6           359         57    416
7            33         57     90
9           117        264    381
10          399        101    500
11           53        165    218
14          287        213    500
15          128         43    171
16          219        142    361
17          124         76    200
18         1201         71   1272
Training cluster models (KNN)…

===== f-SCAN-KNN TEST2 (9:1) RESULTS =====
ACC=0.6509, F1=0.2027, AUC=0.8765, FPR=0.3597
Confusion Matrix:

In [1]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

# ============================================
# 1. Load Dataset
# ============================================

# CSV 불러오기 (sanitize 없음)
train_df = pd.read_csv("CV_train_data.csv")

# label, year 같은 메타데이터 제거 (있을 경우만)
meta_cols = ["label", "year", "apk", "apkname"]
feature_cols = [c for c in train_df.columns if c not in meta_cols]

X = train_df[feature_cols].values   # shape: (N_samples, 1848)

# ============================================
# 2. Scale features (GMM stability에 매우 중요)
# ============================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================
# 3. Run GMM multiple times with different seeds
# ============================================

K = 19           # reviewer가 요구한 K
N_RUNS = 20      # random initialization 반복 횟수

aic_list = []
labels_list = []

for seed in range(N_RUNS):
    print(f"Running GMM K={K}, seed={seed}...")
    
    gmm = GaussianMixture(
        n_components=K,
        covariance_type='diag',   # 중요: full_covariance는 singularity 발생
        max_iter=300,
        random_state=seed
    )
    
    gmm.fit(X_scaled)
    
    # 저장
    aic_list.append(gmm.aic(X_scaled))
    labels_list.append(gmm.predict(X_scaled))

# ============================================
# 4. AIC Variance / Stability
# ============================================

aic_array = np.array(aic_list)
print("\n=== AIC Variance Analysis for K = 19 ===")
print(f"AIC Mean : {aic_array.mean():.4f}")
print(f"AIC Std  : {aic_array.std():.4f}")
print(f"AIC Min  : {aic_array.min():.4f}")
print(f"AIC Max  : {aic_array.max():.4f}")

# ============================================
# 5. Cluster Reproducibility (Adjusted Rand Index)
# ============================================

baseline = labels_list[0]
ari_scores = []

for i in range(1, N_RUNS):
    ari = adjusted_rand_score(baseline, labels_list[i])
    ari_scores.append(ari)

print("\n=== Cluster Reproducibility (ARI) Across 20 Runs ===")
print(f"ARI Mean : {np.mean(ari_scores):.4f}")
print(f"ARI Std  : {np.std(ari_scores):.4f}")
print(f"ARI Min  : {np.min(ari_scores):.4f}")
print(f"ARI Max  : {np.max(ari_scores):.4f}")


Running GMM K=19, seed=0...
Running GMM K=19, seed=1...
Running GMM K=19, seed=2...
Running GMM K=19, seed=3...
Running GMM K=19, seed=4...
Running GMM K=19, seed=5...
Running GMM K=19, seed=6...
Running GMM K=19, seed=7...
Running GMM K=19, seed=8...
Running GMM K=19, seed=9...
Running GMM K=19, seed=10...
Running GMM K=19, seed=11...
Running GMM K=19, seed=12...
Running GMM K=19, seed=13...
Running GMM K=19, seed=14...
Running GMM K=19, seed=15...
Running GMM K=19, seed=16...
Running GMM K=19, seed=17...
Running GMM K=19, seed=18...
Running GMM K=19, seed=19...

=== AIC Variance Analysis for K = 19 ===
AIC Mean : -163625888.7807
AIC Std  : 2991219.7096
AIC Min  : -168909893.6977
AIC Max  : -156506668.1615

=== Cluster Reproducibility (ARI) Across 20 Runs ===
ARI Mean : 0.2763
ARI Std  : 0.0537
ARI Min  : 0.1863
ARI Max  : 0.3751
